In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
with open('Anna.txt', 'r') as file:
    text = file.read()

In [3]:
chars = tuple(set(text))
int_char = dict(enumerate(chars))
char_int = {ch : i for i, ch in int_char.items()}

encoded = np.array([char_int[x] for x in text])
encoded[:15]

array([ 8, 33, 49, 21, 16, 13, 46, 80, 70, 73, 73, 73, 36, 49, 21])

In [4]:
def one_hot_encode(arr, label):
    one_hot = np.array((arr.size, label))
    one_hot[arr.shape[0], arr.flatten()] = 1
    one_hot = one_hot.reshape((*arr.shape,label))
    
    return one_hot

In [14]:
def get_batches(arr, batch_size, seq_len):
    total_seq = batch_size*seq_len
    n_batch = len(arr)//(total_seq)
    arr = arr[:n_batch*(total_seq)]
    arr = arr.reshape((batch_size,-1))

    for n in range(0, arr.shape[1], seq_len):
        x = arr[:, n:n+seq_len]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_len]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [15]:
batches = get_batches(encoded, 8, 50)
x, y = next(batches)

In [17]:
print('x\n', x[:10, :10])
print('y\n', y[:10, :10])

x
 [[ 8 33 49 21 16 13 46 80 70 73]
 [11 71 62 80 16 33 49 16 80 49]
 [13 62 69 80 71 46 80 49 80  6]
 [11 80 16 33 13 80 23 33  1 13]
 [80 11 49 41 80 33 13 46 80 16]
 [23 58 11 11  1 71 62 80 49 62]
 [80 42 62 62 49 80 33 49 69 80]
 [65 34  2 71 62 11 45  4 17 80]]
y
 [[33 49 21 16 13 46 80 70 73 73]
 [71 62 80 16 33 49 16 80 49 16]
 [62 69 80 71 46 80 49 80  6 71]
 [80 16 33 13 80 23 33  1 13  6]
 [11 49 41 80 33 13 46 80 16 13]
 [58 11 11  1 71 62 80 49 62 69]
 [42 62 62 49 80 33 49 69 80 11]
 [34  2 71 62 11 45  4 17 80 37]]
